# Prueba Ingeniero de Datos — Pipeline por micro-batches

**Autor:** Fabian Ramírez · Python · pandas · PostgreSQL (Neon)

Cada CSV se lee **por lotes**. Por cada fila insertada se actualizan las estadísticas de `price` (count, promedio, mínimo, máximo) con acumuladores, sin recalcular sobre los datos ya cargados. Cada lote es **todo o nada**: filas, estadísticas y avance se guardan juntos en una transacción. Así el pipeline puede **reanudarse** tras un fallo y **no duplica** datos si se ejecuta de nuevo.

| Tabla | Contenido |
|---|---|
| `transactions` | Los datos de los CSV |
| `pipeline_stats` | Una fila con los 5 acumuladores: total_filas, n_precios, suma, mínimo, máximo |
| `load_log` | Por archivo: estado (`EN_PROCESO` / `COMPLETADO`) y filas leídas |

## 1. Conexión a PostgreSQL
La cadena de conexión se pide con `getpass`, así la contraseña no queda guardada en el notebook.

In [1]:
!pip install -q "psycopg[binary]"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 34.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.5/215.5 kB 11.7 MB/s eta 0:00:00


In [2]:
import psycopg
import pandas as pd
from getpass import getpass

url = getpass("Pega tu connection string de Neon: ")
conn = psycopg.connect(url, autocommit=True)
print(conn.execute("SELECT version()").fetchone()[0])

Pega tu connection string de Neon: ··········
PostgreSQL 18.6 (6569466) on aarch64-unknown-linux-gnu, compiled by gcc (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0, 64-bit


## 2. Subir los archivos CSV

In [3]:
from google.colab import files
files.upload()

Saving 2012-1.csv to 2012-1.csv
Saving 2012-2.csv to 2012-2.csv
Saving 2012-3.csv to 2012-3.csv
Saving 2012-4.csv to 2012-4.csv
Saving 2012-5.csv to 2012-5.csv
Saving validation.csv to validation.csv


{'2012-1.csv': b'timestamp,price,user_id\r\n1/10/2012,50,9\r\n1/11/2012,87,10\r\n1/12/2012,64,7\r\n1/13/2012,20,10\r\n1/14/2012,14,10\r\n1/15/2012,95,8\r\n1/16/2012,95,2\r\n1/17/2012,62,4\r\n1/18/2012,46,8\r\n1/19/2012,97,2\r\n1/20/2012,31,1\r\n1/21/2012,,2\r\n1/22/2012,37,1\r\n1/23/2012,74,7\r\n1/24/2012,45,5\r\n1/25/2012,27,5\r\n1/26/2012,78,4\r\n1/27/2012,42,6\r\n1/28/2012,45,8\r\n1/29/2012,96,9\r\n1/30/2012,,2\r\n1/31/2012,88,1',
 '2012-2.csv': b'timestamp,price,user_id\r\n2/1/2012,76,1\r\n2/2/2012,66,1\r\n2/3/2012,47,10\r\n2/4/2012,100,5\r\n2/5/2012,19,4\r\n2/6/2012,59,8\r\n2/7/2012,77,3\r\n2/8/2012,100,8\r\n2/9/2012,55,1\r\n2/10/2012,19,2\r\n2/11/2012,95,3\r\n2/12/2012,73,2\r\n2/13/2012,32,1\r\n2/14/2012,95,5\r\n2/15/2012,12,8\r\n2/16/2012,37,8\r\n2/17/2012,28,10\r\n2/18/2012,10,5\r\n2/19/2012,27,9\r\n2/20/2012,14,9\r\n2/21/2012,87,5\r\n2/22/2012,75,7\r\n2/23/2012,36,8\r\n2/24/2012,84,8\r\n2/25/2012,62,6\r\n2/26/2012,56,6\r\n2/27/2012,17,4\r\n2/28/2012,64,7\r\n2/29/2012,68,3',
 '

## 3. Crear las tablas
⚠ Esta celda **borra todo**. Ejecútala solo para empezar de cero; para reanudar una carga, sáltala.

In [4]:
# ⚠ Borra todo y empieza de cero. Ejecutar solo cuando se quiera reiniciar la carga.
conn.execute("DROP TABLE IF EXISTS transactions, pipeline_stats, load_log")

# 1. Datos
conn.execute("""
    CREATE TABLE transactions (
        id          SERIAL PRIMARY KEY,
        event_date  DATE    NOT NULL,
        price       NUMERIC NULL,
        user_id     INTEGER NOT NULL,
        source_file TEXT    NOT NULL
    )
""")

# 2. Estadísticas: una sola fila con los 5 acumuladores
conn.execute("""
    CREATE TABLE pipeline_stats (
        id          INTEGER PRIMARY KEY CHECK (id = 1),
        total_filas INTEGER NOT NULL,
        n_precios   INTEGER NOT NULL,
        suma        NUMERIC NOT NULL,
        minimo      NUMERIC NULL,
        maximo      NUMERIC NULL
    )
""")
conn.execute("INSERT INTO pipeline_stats VALUES (1, 0, 0, 0, NULL, NULL)")

# 3. Registro de carga: qué archivos se procesaron y hasta qué fila
conn.execute("""
    CREATE TABLE load_log (
        archivo      TEXT    PRIMARY KEY,
        estado       TEXT    NOT NULL,     -- 'EN_PROCESO' o 'COMPLETADO'
        filas_leidas INTEGER NOT NULL
    )
""")
print("Tablas creadas: transactions, pipeline_stats, load_log")

Tablas creadas: transactions, pipeline_stats, load_log


## 4. Estadísticas en ejecución
- `actualizar_stats`: suma una fila a los acumuladores (en memoria).
- `guardar_stats_en_bd` / `leer_stats_de_bd`: persisten y recuperan los acumuladores en `pipeline_stats`.

El promedio no se guarda: se calcula como `suma / n_precios`.

In [5]:
def actualizar_stats(price):
    stats["count"] += 1
    if price is None:
        return
    stats["n_precios"] += 1
    stats["suma"] += price
    stats["min"] = price if stats["min"] is None else min(stats["min"], price)
    stats["max"] = price if stats["max"] is None else max(stats["max"], price)

def mostrar_stats():
    promedio = stats["suma"] / stats["n_precios"] if stats["n_precios"] else None
    return (f"count={stats['count']} | promedio={promedio:.4f} | "
            f"min={stats['min']} | max={stats['max']}")

def leer_stats_de_bd():
    """Lee los acumuladores guardados en la base."""
    fila = conn.execute(
        "SELECT total_filas, n_precios, suma, minimo, maximo FROM pipeline_stats WHERE id = 1"
    ).fetchone()
    return {
        "count":     fila[0],
        "n_precios": fila[1],
        "suma":      float(fila[2]),
        "min":       float(fila[3]) if fila[3] is not None else None,
        "max":       float(fila[4]) if fila[4] is not None else None,
    }

def guardar_stats_en_bd():
    """Escribe los acumuladores actuales en la base."""
    conn.execute(
        "UPDATE pipeline_stats SET total_filas = %s, n_precios = %s, suma = %s, minimo = %s, maximo = %s WHERE id = 1",
        (stats["count"], stats["n_precios"], stats["suma"], stats["min"], stats["max"]),
    )

In [6]:
stats = leer_stats_de_bd()          # ya no se empieza en cero: se retoma lo guardado
print("Stats recuperadas de la base:", stats)

Stats recuperadas de la base: {'count': 0, 'n_precios': 0, 'suma': 0.0, 'min': None, 'max': None}


## 5. Pipeline
Por cada archivo:
1. Consulta `load_log`: si está `COMPLETADO` se omite; si está `EN_PROCESO`, se salta lo ya leído.
2. Lee el CSV por lotes (`chunksize`).
3. En **una transacción por lote**: inserta las filas, guarda las stats y suma el avance en `load_log`. Si algo falla, se revierte todo el lote (en la base y en memoria).
4. Al terminar, marca el archivo como `COMPLETADO`.

In [7]:
TAMANO_LOTE = 10

def consultar_log(nombre_archivo):
    """Devuelve (estado, filas_leidas) del archivo, o (None, 0) si nunca se ha cargado."""
    fila = conn.execute(
        "SELECT estado, filas_leidas FROM load_log WHERE archivo = %s", (nombre_archivo,)
    ).fetchone()
    return fila if fila else (None, 0)


def cargar_csv(nombre_archivo, mostrar_filas=False, fallar_en_fila=None):
    # PASO 5: ¿ya se cargó este archivo?
    estado, filas_leidas = consultar_log(nombre_archivo)
    if estado == "COMPLETADO":
        print(f"⏭ {nombre_archivo} ya fue cargado. Se omite.")
        return
    if estado == "EN_PROCESO":
        print(f"↪ {nombre_archivo} quedó a medias: se continúa desde la fila {filas_leidas + 1}")

    lotes = pd.read_csv(nombre_archivo, chunksize=TAMANO_LOTE,
                        skiprows=range(1, filas_leidas + 1))

    for numero_lote, lote in enumerate(lotes, start=1):
        if lote.empty:
            continue
        lote["timestamp"] = pd.to_datetime(lote["timestamp"], format="%m/%d/%Y")

        respaldo = stats.copy()
        try:
            with conn.transaction():                     # TODO O NADA:
                for fila in lote.itertuples(index=False):
                    price = None if pd.isna(fila.price) else float(fila.price)
                    conn.execute(                        #   a) las filas
                        "INSERT INTO transactions (event_date, price, user_id, source_file) VALUES (%s, %s, %s, %s)",
                        (fila.timestamp.date(), price, int(fila.user_id), nombre_archivo),
                    )
                    actualizar_stats(price)
                    if mostrar_filas:
                        print(f"  {nombre_archivo} | lote {numero_lote} | price={price} -> {mostrar_stats()}")
                    if stats["count"] == fallar_en_fila:
                        raise RuntimeError(f"error simulado en la fila {fallar_en_fila}")

                guardar_stats_en_bd()                    #   b) las estadísticas (PASO 4)

                conn.execute("""
                    INSERT INTO load_log (archivo, estado, filas_leidas)
                    VALUES (%s, 'EN_PROCESO', %s)
                    ON CONFLICT (archivo) DO UPDATE
                    SET filas_leidas = load_log.filas_leidas + EXCLUDED.filas_leidas
                """, (nombre_archivo, len(lote)))        #   c) el avance (PASO 5)

        except Exception as error:
            stats.update(respaldo)
            print(f"  ✘ lote {numero_lote} de {nombre_archivo} revertido ({error}) -> {mostrar_stats()}")
            raise

        print(f"  lote {numero_lote} de {nombre_archivo} ({len(lote)} filas) -> {mostrar_stats()}")

    conn.execute("UPDATE load_log SET estado = 'COMPLETADO' WHERE archivo = %s", (nombre_archivo,))
    print(f"✔ {nombre_archivo} cargado -> {mostrar_stats()}")

## 6. Carga de `2012-1.csv` … `2012-5.csv`

In [8]:
for nombre in ["2012-1.csv", "2012-2.csv", "2012-3.csv", "2012-4.csv", "2012-5.csv"]:
    cargar_csv(nombre, mostrar_filas=True)

  2012-1.csv | lote 1 | price=50.0 -> count=1 | promedio=50.0000 | min=50.0 | max=50.0
  2012-1.csv | lote 1 | price=87.0 -> count=2 | promedio=68.5000 | min=50.0 | max=87.0
  2012-1.csv | lote 1 | price=64.0 -> count=3 | promedio=67.0000 | min=50.0 | max=87.0
  2012-1.csv | lote 1 | price=20.0 -> count=4 | promedio=55.2500 | min=20.0 | max=87.0
  2012-1.csv | lote 1 | price=14.0 -> count=5 | promedio=47.0000 | min=14.0 | max=87.0
  2012-1.csv | lote 1 | price=95.0 -> count=6 | promedio=55.0000 | min=14.0 | max=95.0
  2012-1.csv | lote 1 | price=95.0 -> count=7 | promedio=60.7143 | min=14.0 | max=95.0
  2012-1.csv | lote 1 | price=62.0 -> count=8 | promedio=60.8750 | min=14.0 | max=95.0
  2012-1.csv | lote 1 | price=46.0 -> count=9 | promedio=59.2222 | min=14.0 | max=95.0
  2012-1.csv | lote 1 | price=97.0 -> count=10 | promedio=63.0000 | min=14.0 | max=97.0
  lote 1 de 2012-1.csv (10 filas) -> count=10 | promedio=63.0000 | min=14.0 | max=97.0
  2012-1.csv | lote 2 | price=31.0 -> coun

## 7. Resultados después de los 5 archivos

In [9]:
CONSULTA = """
    SELECT COUNT(*)             AS total_filas,
           ROUND(AVG(price), 4) AS promedio,
           MIN(price)           AS minimo,
           MAX(price)           AS maximo
    FROM transactions
"""
print("Stats en ejecución:", mostrar_stats())
antes = pd.DataFrame([conn.execute(CONSULTA).fetchone()], columns=["total_filas", "promedio", "minimo", "maximo"])
antes

Stats en ejecución: count=143 | promedio=57.8849 | min=10.0 | max=100.0


,total_filas,promedio,minimo,maximo
0,143,57.8849,10,100


## 8. Cargar `validation.csv` por el mismo pipeline

In [10]:
cargar_csv("validation.csv", mostrar_filas=True)
print("\nStats en ejecución:", mostrar_stats())

  validation.csv | lote 1 | price=26.0 -> count=144 | promedio=57.6571 | min=10.0 | max=100.0
  validation.csv | lote 1 | price=11.0 -> count=145 | promedio=57.3262 | min=10.0 | max=100.0
  validation.csv | lote 1 | price=13.0 -> count=146 | promedio=57.0141 | min=10.0 | max=100.0
  validation.csv | lote 1 | price=92.0 -> count=147 | promedio=57.2587 | min=10.0 | max=100.0
  validation.csv | lote 1 | price=31.0 -> count=148 | promedio=57.0764 | min=10.0 | max=100.0
  validation.csv | lote 1 | price=62.0 -> count=149 | promedio=57.1103 | min=10.0 | max=100.0
  validation.csv | lote 1 | price=13.0 -> count=150 | promedio=56.8082 | min=10.0 | max=100.0
  validation.csv | lote 1 | price=86.0 -> count=151 | promedio=57.0068 | min=10.0 | max=100.0
  lote 1 de validation.csv (8 filas) -> count=151 | promedio=57.0068 | min=10.0 | max=100.0
✔ validation.csv cargado -> count=151 | promedio=57.0068 | min=10.0 | max=100.0

Stats en ejecución: count=151 | promedio=57.0068 | min=10.0 | max=100.0


## 9. Nueva consulta y comparación

In [11]:
despues = pd.DataFrame([conn.execute(CONSULTA).fetchone()], columns=["total_filas", "promedio", "minimo", "maximo"])
pd.concat([antes, despues], keys=["antes de validation", "después de validation"]).droplevel(1)

,total_filas,promedio,minimo,maximo
antes de validation,143,57.8849,10,100
después de validation,151,57.0068,10,100


In [12]:
# Estado del registro de carga y de las stats guardadas
display(pd.DataFrame(conn.execute("SELECT * FROM load_log ORDER BY archivo").fetchall(),
                     columns=["archivo", "estado", "filas_leidas"]))
pd.DataFrame(conn.execute("SELECT * FROM pipeline_stats").fetchall(),
             columns=["id", "total_filas", "n_precios", "suma", "minimo", "maximo"])

,archivo,estado,filas_leidas
0,2012-1.csv,COMPLETADO,22
1,2012-2.csv,COMPLETADO,29
2,2012-3.csv,COMPLETADO,31
3,2012-4.csv,COMPLETADO,30
4,2012-5.csv,COMPLETADO,31
5,validation.csv,COMPLETADO,8


,id,total_filas,n_precios,suma,minimo,maximo
0,1,151,147,8380,10,100


## 10. Prueba de robustez: fallo, reinicio y reanudación
1. Se reinicia todo y se provoca un error en la fila 37 (lote 2 de `2012-2.csv`).
2. Se simula un reinicio del notebook borrando `stats` de la memoria.
3. Se recuperan las stats desde la base y se reanuda: `2012-1.csv` se omite y `2012-2.csv` continúa desde la fila 11.
4. Se ejecuta todo otra vez: no se duplica nada.

In [13]:
# 1) Empezar de cero
conn.execute("DROP TABLE IF EXISTS transactions, pipeline_stats, load_log")

# 1. Datos
conn.execute("""
    CREATE TABLE transactions (
        id          SERIAL PRIMARY KEY,
        event_date  DATE    NOT NULL,
        price       NUMERIC NULL,
        user_id     INTEGER NOT NULL,
        source_file TEXT    NOT NULL
    )
""")

# 2. Estadísticas: una sola fila con los 5 acumuladores
conn.execute("""
    CREATE TABLE pipeline_stats (
        id          INTEGER PRIMARY KEY CHECK (id = 1),
        total_filas INTEGER NOT NULL,
        n_precios   INTEGER NOT NULL,
        suma        NUMERIC NOT NULL,
        minimo      NUMERIC NULL,
        maximo      NUMERIC NULL
    )
""")
conn.execute("INSERT INTO pipeline_stats VALUES (1, 0, 0, 0, NULL, NULL)")

# 3. Registro de carga: qué archivos se procesaron y hasta qué fila
conn.execute("""
    CREATE TABLE load_log (
        archivo      TEXT    PRIMARY KEY,
        estado       TEXT    NOT NULL,     -- 'EN_PROCESO' o 'COMPLETADO'
        filas_leidas INTEGER NOT NULL
    )
""")
print("Tablas creadas: transactions, pipeline_stats, load_log")

Tablas creadas: transactions, pipeline_stats, load_log


In [14]:
stats = leer_stats_de_bd()
try:
    for nombre in ["2012-1.csv", "2012-2.csv", "2012-3.csv", "2012-4.csv", "2012-5.csv"]:
        cargar_csv(nombre, fallar_en_fila=37)
except RuntimeError as e:
    print("\nPipeline detenido:", e)

  lote 1 de 2012-1.csv (10 filas) -> count=10 | promedio=63.0000 | min=14.0 | max=97.0
  lote 2 de 2012-1.csv (10 filas) -> count=20 | promedio=58.1579 | min=14.0 | max=97.0
  lote 3 de 2012-1.csv (2 filas) -> count=22 | promedio=59.6500 | min=14.0 | max=97.0
✔ 2012-1.csv cargado -> count=22 | promedio=59.6500 | min=14.0 | max=97.0
  lote 1 de 2012-2.csv (10 filas) -> count=32 | promedio=60.3667 | min=14.0 | max=100.0
  ✘ lote 2 de 2012-2.csv revertido (error simulado en la fila 37) -> count=32 | promedio=60.3667 | min=14.0 | max=100.0

Pipeline detenido: error simulado en la fila 37


In [15]:
# 2) Simular reinicio: se pierde todo lo que estaba en memoria
del stats

# 3) Recuperar desde la base y reanudar
stats = leer_stats_de_bd()
print("Stats recuperadas:", stats, "\n")
for nombre in ["2012-1.csv", "2012-2.csv", "2012-3.csv", "2012-4.csv", "2012-5.csv", "validation.csv"]:
    cargar_csv(nombre)

Stats recuperadas: {'count': 32, 'n_precios': 30, 'suma': 1811.0, 'min': 14.0, 'max': 100.0} 

⏭ 2012-1.csv ya fue cargado. Se omite.
↪ 2012-2.csv quedó a medias: se continúa desde la fila 11
  lote 1 de 2012-2.csv (10 filas) -> count=42 | promedio=55.8500 | min=10.0 | max=100.0
  lote 2 de 2012-2.csv (9 filas) -> count=51 | promedio=56.7959 | min=10.0 | max=100.0
✔ 2012-2.csv cargado -> count=51 | promedio=56.7959 | min=10.0 | max=100.0
  lote 1 de 2012-3.csv (10 filas) -> count=61 | promedio=56.4407 | min=10.0 | max=100.0
  lote 2 de 2012-3.csv (10 filas) -> count=71 | promedio=57.8406 | min=10.0 | max=100.0
  lote 3 de 2012-3.csv (10 filas) -> count=81 | promedio=57.8228 | min=10.0 | max=100.0
  lote 4 de 2012-3.csv (1 filas) -> count=82 | promedio=57.9125 | min=10.0 | max=100.0
✔ 2012-3.csv cargado -> count=82 | promedio=57.9125 | min=10.0 | max=100.0
  lote 1 de 2012-4.csv (10 filas) -> count=92 | promedio=57.4778 | min=10.0 | max=100.0
  lote 2 de 2012-4.csv (10 filas) -> count=1

In [16]:
# 4) Re-ejecutar todo: no debe insertar nada
for nombre in ["2012-1.csv", "2012-2.csv", "2012-3.csv", "2012-4.csv", "2012-5.csv", "validation.csv"]:
    cargar_csv(nombre)

print("\nStats en ejecución:", mostrar_stats())
pd.DataFrame([conn.execute(CONSULTA).fetchone()], columns=["total_filas", "promedio", "minimo", "maximo"])

⏭ 2012-1.csv ya fue cargado. Se omite.
⏭ 2012-2.csv ya fue cargado. Se omite.
⏭ 2012-3.csv ya fue cargado. Se omite.
⏭ 2012-4.csv ya fue cargado. Se omite.
⏭ 2012-5.csv ya fue cargado. Se omite.
⏭ validation.csv ya fue cargado. Se omite.

Stats en ejecución: count=151 | promedio=57.0068 | min=10.0 | max=100.0


,total_filas,promedio,minimo,maximo
0,151,57.0068,10,100


## 11. Conclusiones
- Cada CSV se lee por lotes: nunca hay más de `TAMANO_LOTE` filas en memoria.
- Las estadísticas se actualizan después de cada fila con acumuladores (O(1)); la consulta SQL solo se usa para verificar, y ambas coinciden.
- Cada lote es atómico (filas + stats + avance), así que tras un fallo nada queda a medias.
- Las stats viven en la base: sobreviven a un reinicio del notebook.
- `load_log` permite reanudar desde el último lote confirmado y evita duplicados al re-ejecutar.
- Resultados: 143 filas y promedio 57.8849 tras los 5 archivos; 151 filas y promedio 57.0068 tras `validation.csv`. Mínimo 10 y máximo 100 en ambos casos.